# LangChain Memory 快速入门

当前 notebook 用来验证最小 session-only weather memory 链路：

1. 读取并校验环境变量
2. 验证同一 session 能承接上一轮上下文
3. 验证不同 session 不串话
4. 验证 clear 后 memory 失效
5. 对比无 memory 的 agent 行为


In [7]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config.env import load_env
from src.chains.agent_chain import run_weather_agent
from src.chains.memory_chain import clear_weather_agent_memory, run_weather_agent_with_memory


In [8]:
env = load_env()
print({
    "ollama_base_url": env.ollama_base_url,
    "ollama_model": env.ollama_model,
    "weather_api_key_configured": bool(env.weather_api_key),
})


{'ollama_base_url': 'http://localhost:11434', 'ollama_model': 'qwen3:8b', 'weather_api_key_configured': True}


In [9]:
session_id = "weather-memory-demo"
clear_weather_agent_memory(session_id)

first_result = run_weather_agent_with_memory(session_id, "我在北京")
print(first_result["history_used"])
print(first_result["message_count"])
print(first_result["final_answer"])

second_result = run_weather_agent_with_memory(session_id, "现在天气怎么样？")
print(second_result["history_used"])
print(second_result["used_tools"])
print(second_result["tool_names"])
print(second_result["intermediate_steps"])
print(second_result["message_count"])
print(second_result["final_answer"])


False
4
你好呀，在北京有什么计划吗？现在是3月30日早上9点14分，北京当前天气局部多云，气温13.3°C，体感温度12.8°C，空气湿度25%，风力8.6公里/小时。这个时节昼夜温差有点大，注意添减衣物哦~
True
True
['get_weather']
[{'round': 1, 'tool_name': 'get_weather', 'tool_args': {'city': '北京'}, 'tool_output': '北京，Beijing，中国；当地时间 2026-03-30 09:14；天气 局部多云；当前温度 13.3°C；体感温度 12.8°C；湿度 25%；风速 8.6 km/h。'}]
8
你好呀，现在北京的天气是局部多云，当前温度13.3°C，体感温度12.8°C，空气湿度25%，风力8.6公里/小时。这个时节的北京早晚温差较大，出门记得根据时段调整衣物哦~


In [10]:
session_a = "weather-memory-shanghai"
session_b = "weather-memory-guangzhou"
clear_weather_agent_memory(session_a)
clear_weather_agent_memory(session_b)

run_weather_agent_with_memory(session_a, "我在上海")
run_weather_agent_with_memory(session_b, "我在广州")

session_a_result = run_weather_agent_with_memory(session_a, "现在天气怎么样？")
session_b_result = run_weather_agent_with_memory(session_b, "现在天气怎么样？")
print(session_a_result["intermediate_steps"])
print(session_a_result["final_answer"])
print(session_b_result["intermediate_steps"])
print(session_b_result["final_answer"])


[{'round': 1, 'tool_name': 'get_weather', 'tool_args': {'city': '上海'}, 'tool_output': '上海，Shanghai，中国；当地时间 2026-03-30 09:34；天气 小雨；当前温度 15.5°C；体感温度 15.5°C；湿度 97%；风速 20.5 km/h。'}]
上海现在天气小雨，当前温度为15.5°C，体感温度同样为15.5°C，湿度较高达到97%，风速约20.5公里/小时。建议外出携带雨具哦！
[]
当前时间：2026年3月30日 09:15，广州（广东省，中国）。天气情况：局部地区有零星小雨，当前气温24.2°C，体摘温度25.7°C，空气湿度89%，风速14.8 km/h。建议外出携带雨具，注意防雨防滑。


In [11]:
clear_weather_agent_memory(session_id)
cleared_result = run_weather_agent_with_memory(session_id, "现在天气怎么样？")
print(cleared_result["history_used"])
print(cleared_result["used_tools"])
print(cleared_result["tool_names"])
print(cleared_result["final_answer"])


False
False
[]
请问您想查询哪个城市的天气呢？


In [12]:
print(run_weather_agent("我在北京"))
print(run_weather_agent("现在天气怎么样？"))


{'input': '我在北京', 'used_tools': True, 'tool_names': ['get_weather'], 'intermediate_steps': [{'round': 1, 'tool_name': 'get_weather', 'tool_args': {'city': '北京'}, 'tool_output': '北京，Beijing，中国；当地时间 2026-03-30 09:14；天气 局部多云；当前温度 13.3°C；体感温度 12.8°C；湿度 25%；风速 8.6 km/h。'}], 'final_answer': '北京当前天气局部多云，温度13.3°C，体感温度12.8°C，湿度25%，风速8.6 km/h。空气质量良好，建议可适当准备外套应对温差。'}
{'input': '现在天气怎么样？', 'used_tools': False, 'tool_names': [], 'intermediate_steps': [], 'final_answer': '请问您所在的城市是哪里？这样我可以为您查询实时天气情况。'}
